In [3]:
import json
import pandas as pd
from collections import defaultdict
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

DATA_DIR = Path("../data")  # if notebook is inside notebooks folder

INPUT_FILE = DATA_DIR / "all_files_extracted_data.json"
OUTPUT_FILE = DATA_DIR / "processed_workforce_data.json"

# ============================================================
# LOAD EXTRACTED JSON
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

workbook = data["Wohlig Active Employee Data.xlsx"]

active_data = workbook["Active"]

# ============================================================
# ACTIVE DATAFRAME
# ============================================================

active_df = pd.DataFrame(active_data)

active_df.columns = [
    "sr_no",
    "employee_name",
    "doj",
    "role",
    "project",
    "location",
    "project_start",
    "project_end",
    "manager"
]

# Remove header row
active_df = active_df[
    active_df["employee_name"] != "Active Employee"
].reset_index(drop=True)

# ============================================================
# BUILD EMPLOYEE STRUCTURE
# ============================================================

employees = {}
current_employee = None

for _, row in active_df.iterrows():

    employee_name = row["employee_name"]

    # Main Employee Row
    if pd.notna(employee_name):

        current_employee = str(employee_name).strip()

        employees[current_employee] = {
            "name": current_employee,
            "role": str(row["role"]).strip(),
            "manager": str(row["manager"]).strip(),
            "allocations": []
        }

        if pd.notna(row["project"]):
            employees[current_employee]["allocations"].append(
                str(row["project"]).strip()
            )

    # Continuation Row
    else:

        if current_employee and pd.notna(row["project"]):
            employees[current_employee]["allocations"].append(
                str(row["project"]).strip()
            )

# ============================================================
# MULTI ALLOCATED
# ============================================================

multi_allocated = [
    emp
    for emp in employees.values()
    if len(emp["allocations"]) > 1
]
multi_allocated_employee_list = [
    emp["name"]
    for emp in multi_allocated
]

# ============================================================
# UNALLOCATED
# ============================================================

UNALLOCATED_EXCLUSIONS = {
    "samay gada",
    "bilal sani"
}

unallocated = []

for emp in employees.values():

    allocations = [
        str(a).strip().lower()
        for a in emp["allocations"]
    ]

    if (
        len(allocations) == 0
        or allocations == ["no task"]
        or allocations == [""]
    ):

        if emp["name"].strip().lower() not in UNALLOCATED_EXCLUSIONS:
            unallocated.append(emp)


# ============================================================
# BUILD CATEGORY MAPPING FROM SHEETS
# ============================================================

classification_map = {}
closed_projects = []
completed_projects = []
active_projects = []
active_retainers = []
active_internal = []

# -------------------------
# PROJECT SHEET
# -------------------------

for row in workbook["Project"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status == "closed":
        closed_projects.append(project_name)

    elif status == "done":
        completed_projects.append(project_name)

    else:
        classification_map[project_name.lower()] = "Project"
        active_projects.append(project_name)

# -------------------------
# RETAINER SHEET
# -------------------------

for row in workbook["Retainers"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status != "closed":
        classification_map[project_name.lower()] = "Retainer"
        active_retainers.append(project_name)

# -------------------------
# INTERNAL SHEET
# -------------------------

for row in workbook["Internal"]:

    project_name = row.get("Project Name")
    status = str(row.get("Status", "")).strip().lower()

    if not project_name:
        continue

    project_name = str(project_name).strip()

    if status == "closed":

        closed_projects.append(project_name)

    else:

        classification_map[project_name.lower()] = "Internal"
        active_internal.append(project_name)


# ============================================================
# CLOSED PROJECT LOOKUP
# ============================================================

closed_projects_lower = {
    project.lower()
    for project in closed_projects
}

# ============================================================
# CATEGORY COUNTS
# ============================================================

project_count = 0
retainer_count = 0
internal_count = 0

for emp in employees.values():

    allocations = [
        str(a).strip().lower()
        for a in emp["allocations"]
    ]

    if allocations == ["no task"] or len(allocations) == 0:
        continue

    is_project = False
    is_retainer = False
    is_internal = False

    for allocation in allocations:

        category = classification_map.get(allocation)

        if category == "Project":
            is_project = True

        elif category == "Retainer":
            is_retainer = True

        elif category == "Internal":
            is_internal = True

    if is_project:
        project_count += 1

    if is_retainer:
        retainer_count += 1

    if is_internal:
        internal_count += 1

current_projects = (
    len(active_projects)
    + len(active_retainers)
    + len(active_internal)
)
# ============================================================
# WORKFORCE OVERVIEW
# ============================================================


workforce_overview = {

    "active_employees": len(employees),

    "current_projects": current_projects,

    "unallocated_employees": len(unallocated),

    "project_distribution": {

        "client_projects": {
            "count": len(active_projects),
            "projects": sorted(active_projects)
        },

        "retainer_projects": {
            "count": len(active_retainers),
            "projects": sorted(active_retainers)
        },

        "internal_projects": {
            "count": len(active_internal),
            "projects": sorted(active_internal)
        }
    }
}
# ============================================================
# UNALLOCATED EMPLOYEE LIST
# ============================================================

unallocated_employee_list = []

for emp in unallocated:

    unallocated_employee_list.append({
        "name": emp["name"],
        "current_role": emp["role"],
        "reporting_to": emp["manager"]
    })

# ============================================================
# PROJECT ALLOCATION SUMMARY
# ============================================================

allocation_summary = defaultdict(list)

for emp in employees.values():

    for allocation in emp["allocations"]:

        allocation_summary[allocation].append(
            emp["name"]
        )

project_allocation_summary = []

NORMALIZATION = {
    "apollo hispital": "apollo hospital"
}

closed_projects_lower = {
    p.strip().lower()
    for p in closed_projects
}

completed_projects_lower = {
    p.strip().lower()
    for p in completed_projects
}

for project_name, employee_names in allocation_summary.items():

    normalized_name = NORMALIZATION.get(
        project_name.strip().lower(),
        project_name.strip().lower()
    )

    # Skip No Task
    if normalized_name == "no task":
        continue

    # Skip Completed Projects
    if normalized_name in completed_projects_lower:
        continue

    # Skip Closed Projects
    if normalized_name in closed_projects_lower:
        continue

    project_allocation_summary.append({
        "project_name": project_name,
        "employee_count": len(employee_names),
        "employees": sorted(employee_names)
    })

    project_allocation_summary = sorted(
    project_allocation_summary,
    key=lambda x: x["employee_count"],
    reverse=True
)

# ============================================================
# FINAL OUTPUT
# ============================================================

processed_data = {
    "workforce_overview": workforce_overview,
    "unallocated_employee_list": unallocated_employee_list,
    "project_allocation_summary": project_allocation_summary
}

# ============================================================
# SAVE JSON
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        processed_data,
        f,
        indent=4,
        ensure_ascii=False
    )

# ============================================================
# DISPLAY
# ============================================================

print("\n========== WORKFORCE OVERVIEW ==========")
print(json.dumps(workforce_overview, indent=4))

print("\n========== UNALLOCATED EMPLOYEES ==========")

for emp in unallocated_employee_list:
    print(emp)

print("\n========== PROJECT ALLOCATION SUMMARY ==========")

for project in project_allocation_summary:
    print(
        f"{project['project_name']} -> "
        f"{project['employee_count']} employees"
    )

print(f"\nSaved: {OUTPUT_FILE}")


========== WORKFORCE OVERVIEW ==========
{
    "active_employees": 31,
    "current_projects": 8,
    "unallocated_employees": 5,
    "project_distribution": {
        "client_projects": {
            "count": 1,
            "projects": [
                "Mahindra CFO"
            ]
        },
        "retainer_projects": {
            "count": 5,
            "projects": [
                "IKS SRE",
                "JM Finance",
                "Jindal Steel",
                "TUI",
                "Vaultify"
            ]
        },
        "internal_projects": {
            "count": 2,
            "projects": [
                "E-mail Automation",
                "Weekly Automation Task"
            ]
        }
    }
}

========== UNALLOCATED EMPLOYEES ==========
{'name': 'Siddesh Kale', 'current_role': 'Software Developer', 'reporting_to': 'Dhruv Gansinghani'}
{'name': 'Nishant Patil', 'current_role': 'Software Developer', 'reporting_to': 'Dhruv Gansinghani'}
{'name': 'Chintan Limb

In [11]:
import os
import json
import re
from ollama import Client

# ==========================================
# CONFIGURATION & CLIENT SETUP
# ==========================================
MODEL_NAME = "gemma4:31b-cloud"
INPUT_FILE = "../data/all_files_extracted_data.json"
OUTPUT_FILE = "../data/workforce_analysis_output.json"

_api_key = os.environ.get('OLLAMA_API_KEY') or ""
if not _api_key:
    print("[WARNING] OLLAMA_API_KEY is not set. Requests will likely fail with 401.")

client = Client(
    host="https://ollama.com",
    headers={'Authorization': f'Bearer {_api_key}'}
)
# ==========================================


def load_input_data(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find {filepath}. Please ensure the file exists.")
        exit(1)
    except json.JSONDecodeError:
        print(f"Error: {filepath} is not valid JSON.")
        exit(1)


# def build_prompt(json_data):
#     data_string = json.dumps(json_data, indent=2)

#     prompt = f"""
# You are an Expert HR & Project Allocation Auditor. I am providing you with the complete extracted JSON data from a Workforce system.

# Your objective is to reconcile this data by applying a strict, step-by-step logical thinking process. You MUST build the lists first, apply role filters, resolve typos and ghost employees, filter inactive projects, check for multi-allocations, and then calculate accurate final totals.

# ### ROBUST LOGICAL THINKING FRAMEWORK:
# You MUST follow this exact sequence in your `step_by_step_reasoning` section:

# * **Step 1: Master Employee Extraction & Role Filtering:**
#   Scan the Active, Retainer, and Internal sheets. Extract all employee names.
#   CRITICAL RULE: You MUST include employees whose role contains "Intern".
#   You MUST EXCLUDE any employee whose role contains "Tech Lead".
#   Deduplicate this master list so no name appears more than once.
#   Log every exclusion in `audit_logs`.

# * **Step 2: Ghost & Typo Resolution:**
#   Map all names from project sheets back against the Step 1 filtered master list.
#   Resolve typos (e.g., "Apollo Hispital" -> "Apollo Hospital").
#   Identify ghost employees (appear in project sheets but not in the master list) and discard them.
#   Resolve empty rows in the Active sheet by assigning the project data to the last named employee above them.

# * **Step 3: Status Filtering:**
#   Check the "Status" column for ALL Projects, Retainers, and Internal tasks.
#   If the Status is "Done", "Closed", or "Finished", completely EXCLUDE that entry from active arrays.
#   Track excluded entries in `workforce_overview.completed_project_names` and `closed_project_names`.

# * **Step 4: Build Active Allocation Lists:**
#   Construct final arrays for active `projects`, `retainers`, and `internal` tasks — containing only entries with active statuses and valid (non-Tech-Lead, non-ghost) employees.
#   Additionally, compile a unified `active_projects_table` that consolidates all three types into a single flat list with a `project_type` field for each entry.

# * **Step 5: Build Unallocated List:**
#   Build the complete list of `unallocated_employees` — those in the Active sheet with an empty project, "No Task", or "Bench".
#   Apply the Tech Lead exclusion rule here as well.

# * **Step 6: Multi-Allocation Cross-Check (CRITICAL):**
#   Scan every employee name inside your finalized active `projects`, `retainers`, and `internal` lists.
#   If an employee appears MORE THAN ONCE across these lists combined (e.g., in a Retainer AND an Internal task), they are multi-allocated.
#   List their names and all the projects they appear in. State the exact total count.

# * **Step 7: Calculate Final Totals:**
#   Sum all counts strictly from the arrays generated above.
#   `total_multi_allocated_employees` MUST equal the count from Step 6.
#   `total_filtered_employees` MUST equal the count from the deduplicated Step 1 master list.

# ### CORE AUDIT RULES:
# - The "Active Employee" data is the absolute source of truth.
# - Tech Leads MUST be excluded from ALL final employee counts, project assignments, and unallocated lists.
# - Interns are explicitly treated as active employees and must be included.
# - Projects with "Done" or "Closed" Status must ONLY appear in completed/closed counters — NOT in any active allocation array.

# ### REQUIRED JSON OUTPUT SCHEMA:
# Return ONLY a valid JSON object matching the exact structure below. No markdown, no extra text.

# {{
#   "step_by_step_reasoning": {{
#     "step_1_extraction_and_role_filtering": "Detail the extraction, inclusion of Interns, explicit exclusion of Tech Leads, and deduplication.",
#     "step_2_ghosts_and_typos": "Describe typo fixes, ghost removals, and empty row resolutions.",
#     "step_3_status_filtering": "List all projects/retainers/internal tasks marked Done or Closed that are excluded.",
#     "step_4_build_active_allocations": "Confirm the active projects, retainers, internal tasks, and the unified active_projects_table.",
#     "step_5_build_unallocated": "List all unallocated employees (after Tech Lead exclusion). State the total count.",
#     "step_6_multi_allocation_cross_check": "List every employee appearing in more than one active list, and which lists. State the exact count.",
#     "step_7_calculate_totals": "Show the arithmetic for all totals. Confirm multi-allocated and filtered totals match Steps 6 and 1."
#   }},
#   "active_employees_filtered_list": ["string"],
#   "unallocated_employees": [
#     {{
#       "name": "string",
#       "current_role": "string",
#       "reporting_to": "string"
#     }}
#   ],
#   "project_allocations": {{
#     "projects": [
#       {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
#     ],
#     "retainers": [
#       {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
#     ],
#     "internal": [
#       {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
#     ]
#   }},
#   "active_projects_table": [
#     {{
#       "project_name": "string",
#       "project_type": "Project | Retainer | Internal",
#       "status": "string",
#       "employee_count": 0,
#       "employees": ["string"]
#     }}
#   ],
#   "workforce_overview": {{
#     "total_filtered_employees": 0,
#     "total_unallocated_employees": 0,
#     "total_multi_allocated_employees": 0,
#     "multi_allocated_names": ["string"],
#     "total_active_projects": 0,
#     "total_project_type_employees": 0,
#     "total_retainer_type_employees": 0,
#     "total_internal_type_employees": 0,
#     "completed_projects_count": 0,
#     "completed_project_names": ["string"],
#     "closed_projects_count": 0,
#     "closed_project_names": ["string"]
#   }},
#   "audit_logs": [
#     "string (e.g., 'ROLE FILTER: Excluded [Name] — role is Tech Lead.' or 'MULTI-ALLOCATION: [Name] found in both Retainers and Internal.')"
#   ]
# }}

# ### DATA INPUT (JSON format):
# {data_string}
# """
#     return prompt

def build_prompt(json_data):
    data_string = json.dumps(json_data, indent=2)

    prompt = f"""
You are an Expert HR & Project Allocation Auditor. I am providing you with the complete extracted JSON data from a Workforce system.

Your objective is to reconcile this data by applying a strict, step-by-step logical thinking process. You MUST build the lists first, apply role filters, resolve typos and ghost employees, filter inactive projects, check for multi-allocations, and then calculate accurate final totals.

### ROBUST LOGICAL THINKING FRAMEWORK:
You MUST follow this exact sequence in your `step_by_step_reasoning` section:

* **Step 1: Master Employee Extraction & Role Filtering:**
  Scan the Active, Retainer, and Internal sheets. Extract all employee names.
  CRITICAL RULE: You MUST include employees whose role contains "Intern".
  You MUST include employees whose role contains "Tech Lead" in this master list.
  Deduplicate this master list so no name appears more than once.

* **Step 2: Ghost & Typo Resolution:**
  Map all names from project sheets back against the Step 1 filtered master list.
  Resolve typos (e.g., "Apollo Hispital" -> "Apollo Hospital").
  Identify ghost employees (appear in project sheets but not in the master list) and discard them.
  Resolve empty rows in the Active sheet by assigning the project data to the last named employee above them.

* **Step 3: Status Filtering:**
  Check the "Status" column for ALL Projects, Retainers, and Internal tasks.
  If the Status is "Done", "Closed", or "Finished", completely EXCLUDE that entry from active arrays.
  Track excluded entries in `workforce_overview.completed_project_names` and `closed_project_names`.

* **Step 4: Build Active Allocation Lists:**
  Construct final arrays for active `projects`, `retainers`, and `internal` tasks — containing only entries with active statuses and valid (non-ghost) employees.
  Additionally, compile a unified `active_projects_table` that consolidates all three types into a single flat list with a `project_type` field for each entry.

* **Step 5: Build Unallocated List:**
  Build the complete list of `unallocated_employees` — those in the Active sheet with an empty project, "No Task", or "Bench".
  CRITICAL RULE: You MUST EXCLUDE any employee whose role contains "Tech Lead" from this unallocated list.
  Log every such exclusion in `audit_logs`.

* **Step 6: Multi-Allocation Cross-Check (CRITICAL):**
  Scan every employee name inside your finalized active `projects`, `retainers`, and `internal` lists.
  If an employee appears MORE THAN ONCE across these lists combined (e.g., in a Retainer AND an Internal task), they are multi-allocated.
  List their names and all the projects they appear in. State the exact total count.

* **Step 7: Calculate Final Totals:**
  Sum all counts strictly from the arrays generated above.
  `total_multi_allocated_employees` MUST equal the count from Step 6.
  `total_filtered_employees` MUST equal the count from the deduplicated Step 1 master list.

### CORE AUDIT RULES:
- The "Active Employee" data is the absolute source of truth.
- Tech Leads MUST be excluded ONLY from the `unallocated_employees` list. They remain in all other counts and project assignments.
- Interns are explicitly treated as active employees and must be included.
- Projects with "Done" or "Closed" Status must ONLY appear in completed/closed counters — NOT in any active allocation array.

### REQUIRED JSON OUTPUT SCHEMA:
Return ONLY a valid JSON object matching the exact structure below. No markdown, no extra text.

{{
  "step_by_step_reasoning": {{
    "step_1_extraction_and_role_filtering": "Detail the extraction, inclusion of Interns, inclusion of Tech Leads in the master list, and deduplication.",
    "step_2_ghosts_and_typos": "Describe typo fixes, ghost removals, and empty row resolutions.",
    "step_3_status_filtering": "List all projects/retainers/internal tasks marked Done or Closed that are excluded.",
    "step_4_build_active_allocations": "Confirm the active projects, retainers, internal tasks, and the unified active_projects_table.",
    "step_5_build_unallocated": "List all unallocated employees (after Tech Lead exclusion). State the total count.",
    "step_6_multi_allocation_cross_check": "List every employee appearing in more than one active list, and which lists. State the exact count.",
    "step_7_calculate_totals": "Show the arithmetic for all totals. Confirm multi-allocated and filtered totals match Steps 6 and 1."
  }},
  "active_employees_filtered_list": ["string"],
  "unallocated_employees": [
    {{
      "name": "string",
      "current_role": "string",
      "reporting_to": "string"
    }}
  ],
  "project_allocations": {{
    "projects": [
      {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
    ],
    "retainers": [
      {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
    ],
    "internal": [
      {{ "project_name": "string", "employee_count": 0, "employees": ["string"] }}
    ]
  }},
  "active_projects_table": [
    {{
      "project_name": "string",
      "project_type": "Project | Retainer | Internal",
      "status": "string",
      "employee_count": 0,
      "employees": ["string"]
    }}
  ],
  "workforce_overview": {{
    "total_filtered_employees": 0,
    "total_unallocated_employees": 0,
    "total_multi_allocated_employees": 0,
    "multi_allocated_names": ["string"],
    "total_active_projects": 0,
    "total_project_type_employees": 0,
    "total_retainer_type_employees": 0,
    "total_internal_type_employees": 0,
    "completed_projects_count": 0,
    "completed_project_names": ["string"],
    "closed_projects_count": 0,
    "closed_project_names": ["string"]
  }},
  "audit_logs": [
    "string (e.g., 'ROLE FILTER: Excluded [Name] from unallocated — role is Tech Lead.' or 'MULTI-ALLOCATION: [Name] found in both Retainers and Internal.')"
  ]
}}

### DATA INPUT (JSON format):
{data_string}
"""
    return prompt


def extract_json_from_response(response_text):
    match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if match:
        return match.group(0)
    return response_text


def run_audit():
    print("Loading extracted JSON data...")
    input_data = load_input_data(INPUT_FILE)

    prompt = build_prompt(input_data)

    print(f"Sending data to {MODEL_NAME} via configured Cloud Client...")
    print("Applying Role Filters (Excluding Tech Leads, Including Interns)...")
    print("Executing Deep Cross-Check for Multi-Allocated Employees...")

    try:
        response = client.generate(
            model=MODEL_NAME,
            prompt=prompt,
            format='json',
            options={
                'temperature': 0.1
            }
        )

        result_text = response.get('response', '')

        clean_json_string = extract_json_from_response(result_text)
        parsed_json = json.loads(clean_json_string)

        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(parsed_json, f, indent=2)

        print(f"Success! Full audit report saved to {OUTPUT_FILE}")

    except Exception as e:
        print(f"An error occurred during API execution: {e}")


if __name__ == "__main__":
    run_audit()

Loading extracted JSON data...
Sending data to gemma4:31b-cloud via configured Cloud Client...
Applying Role Filters (Excluding Tech Leads, Including Interns)...
Executing Deep Cross-Check for Multi-Allocated Employees...
Success! Full audit report saved to ../data/workforce_analysis_output.json


In [10]:
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path("../.env"))

print("KEY:", os.getenv("OLLAMA_API_KEY"))
print("FILE EXISTS:", os.path.exists("../data/all_files_extracted_data.json"))

KEY: 0117d84d72b94e9aa9b5fdc174810c6f.hpNYLqKbi0jkWSWX9LKT-DhG
FILE EXISTS: True
